# "THE PRICE IS RIGHT" Capstone Project

This week - build a model that predicts how much something costs from a description, based on a scrape of Amazon data


A model that can estimate how much something costs, from its description.

# Order of play

DAY 1: Data Curation  
DAY 2: Data Pre-processing  
DAY 3: Evaluation, Baselines, Traditional ML  
DAY 4: Deep Learning and LLMs  
DAY 5: Fine-tuning a Frontier Model  

## DAY 2: Data Pre-processing

Today we'll rewrite the products into a standard format.  
LLMs are great at this!


<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business value of Data Pre-processing / Re-writing</h2>
            <span style="color:#181;">LLMs have made it simple to do something that was considered impossible only a few years ago.
            This approach can be applied to almost any business vertical, and it's similar to the advanced techniques
            we used on Week 5.</span>
        </td>
    </tr>
</table>

In [3]:
from litellm import completion
from dotenv import load_dotenv
import json
from pricer.batch import Batch
from pricer.items import Item
import os

load_dotenv(override=True)
hf_key = os.getenv("HF-TOKEN")

# 3. Add a quick test to prove the key is loaded
if not hf_key:
    print("WARNING: Python did not find the HF_TOKEN in the .env file!")
else:
    print(f"Token loaded successfully! Starts with: {hf_key[:4]}")

Token loaded successfully! Starts with: hf_Z


# The next cell is where you choose Dataset

Use `LITE_MODE = True` for the free, fast version with training data size of 20,000

USe `LITE_MODE =  False` for the powerful, full version with training data size of 800,000

## For this lab

You can skip altogether and load the dataset from HuggingFace: $0

You can run pre-processing for the lite dataset: under $1

You can run pre-processing for the full dataset: $30

In [4]:
LITE_MODE = True

In [5]:
username = "shweta8961"
dataset = f"{username}/items_raw_lite" if LITE_MODE else f"{username}/items_raw_full"

train, val, test = Item.from_hub(dataset)

items = train + val + test

print(f"Loaded {len(items):,} items")
print(items[0])

README.md:   0%|          | 0.00/736 [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


train-00000-of-00001.parquet:   0%|          | 0.00/9.74M [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


validation-00000-of-00001.parquet:   0%|          | 0.00/731k [00:00<?, ?B/s]

...

test-00000-of-00001.parquet:   0%|          | 0.00/729k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/800 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/800 [00:00<?, ? examples/s]

Loaded 11,600 items
title='Proline PLJW 120 Under Cabinet Range Hood - 6 Speed - 900 Max CFM Low Profile - Stainless Steel Professional Baffle Filters Dishwasher safe 3 Year Warranty Sizes include 30 36 42 and 48 inch' category='Appliances' price=989.99 full='Proline PLJW 120 Under Cabinet Range Hood - 6 Speed - 900 Max CFM Low Profile - Stainless Steel Professional Baffle Filters Dishwasher safe 3 Year Warranty Sizes include 30 36 42 and 48 inch\n[\'Stainless Baffle Filters that are easy to remove, and the quietest 385 CFM setting in the industry. (Based on comparable size and dual local blower capacity). 900 CFM total capacity with Elegant and Efficient "Time Delay" Touch Controls. This Range Hood comes with blower and fan completely installed, and factory tested. This makes the installation one of the easiest in the industry. Specifications: 110v 60hz (USA and Canada Certification) Electronic Controls 6 Speed, 900 Max. CFM Blower LED Lights Beautiful Low Profile Design Stainless Baf

In [6]:
items[2].id

In [7]:
# Give every item an id

for index, item in enumerate(items):
    item.id = index


In [8]:


SYSTEM_PROMPT = """Create a concise description of a product. Respond only in this format. Do not include part numbers.
Title: Rewritten short precise title
Category: eg Electronics
Brand: Brand name
Description: 1 sentence description
Details: 1 sentence on features"""

In [9]:
print(items[0].full)

Proline PLJW 120 Under Cabinet Range Hood - 6 Speed - 900 Max CFM Low Profile - Stainless Steel Professional Baffle Filters Dishwasher safe 3 Year Warranty Sizes include 30 36 42 and 48 inch
['Stainless Baffle Filters that are easy to remove, and the quietest 385 CFM setting in the industry. (Based on comparable size and dual local blower capacity). 900 CFM total capacity with Elegant and Efficient "Time Delay" Touch Controls. This Range Hood comes with blower and fan completely installed, and factory tested. This makes the installation one of the easiest in the industry. Specifications: 110v 60hz (USA and Canada Certification) Electronic Controls 6 Speed, 900 Max. CFM Blower LED Lights Beautiful Low Profile Design Stainless Baffle Filters Easy to Install, Everything Included Seamless Design Easy to Clean and Maintain Brushed Stainless Steel Product Weight 65 lbs Duct Vent Size: 7" Dimensions: 35 3/4" wide x 22" deep x 10" tall']
[' BLOWER WITH A 900 CFM MAX. OUTPUT - Industrial streng

In [ ]:
# donot run

messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": items[0].full}]
response = completion(messages=messages, model="groq/openai/gpt-oss-20b", reasoning_effort="low")
print(response.choices[0].message.content)
print()
print(f"Input tokens: {response.usage.prompt_tokens}")
print(f"Output tokens: {response.usage.completion_tokens}")
print(f"Cost: {response._hidden_params['response_cost']*100:.3f} cents")



Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


Provider List: https://docs.litellm.ai/docs/providers



AuthenticationError: litellm.AuthenticationError: HuggingfaceException - <!DOCTYPE html>
<html class="" lang="en">
<head>
    <meta charset="utf-8" />
    <meta
            name="viewport"
            content="width=device-width, initial-scale=1.0, user-scalable=no"
    />
    <meta
            name="description"
            content="We're on a journey to advance and democratize artificial intelligence through open source and open science."
    />
    <meta property="fb:app_id" content="1321688464574422" />
    <meta name="twitter:card" content="summary_large_image" />
    <meta name="twitter:site" content="@huggingface" />
    <meta
            property="og:title"
            content="Hugging Face - The AI community building the future."
    />
    <meta property="og:type" content="website" />

    <title>Hugging Face - The AI community building the future.</title>
    <style>
        body {
            margin: 0;
        }

        main {
            background-color: white;
        

In [12]:

messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": items[0].full}]
response = completion(messages=messages, model="ollama/llama3.2:1b", api_base="http://localhost:11434")
print(response.choices[0].message.content)
print()
print(f"Input tokens: {response.usage.prompt_tokens}")
print(f"Output tokens: {response.usage.completion_tokens}")
print(f"Cost: {response._hidden_params['response_cost']*100:.3f} cents")


**Rewritten Short Precise Title**
Proline Under Cabinet Range Hood - 6 Speed - Low Profile Stainless Steel Professional Baffle Filters Dishwasher Safe

**Category: Electronics**
Proline
**Description:** A high-performance under-cabinet range hood with 900 CFM maximum airflow, stainless steel baffle filters for easy cleaning, and a quiet operation of 385 CFM.
**Details:** Features include low profile design, electronic controls, and dishwasher-safe stainless steel baffle filters that are easy to remove.

Input tokens: 688
Output tokens: 104
Cost: 0.000 cents


In [14]:
# MODEL = "openai/gpt-oss-20b"
MODEL = "llama:3.2:1b"
items[0]

<Proline PLJW 120 Under Cabinet Range Hood - 6 Speed - 900 Max CFM Low Profile - Stainless Steel Professional Baffle Filters Dishwasher safe 3 Year Warranty Sizes include 30 36 42 and 48 inch = $989.99>

In [16]:
def make_jsonl(item):
    body = {"model": MODEL, "messages": [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": item.full}], "reasoning_effort": "low"}
    line = {"custom_id": str(item.id), "method": "POST", "url": "/v1/chat/completions", "body": body}
    return json.dumps(line)

In [17]:
items[0]

<Proline PLJW 120 Under Cabinet Range Hood - 6 Speed - 900 Max CFM Low Profile - Stainless Steel Professional Baffle Filters Dishwasher safe 3 Year Warranty Sizes include 30 36 42 and 48 inch = $989.99>

In [18]:
make_jsonl(items[0])

'{"custom_id": "0", "method": "POST", "url": "/v1/chat/completions", "body": {"model": "llama:3.2:1b", "messages": [{"role": "system", "content": "Create a concise description of a product. Respond only in this format. Do not include part numbers.\\nTitle: Rewritten short precise title\\nCategory: eg Electronics\\nBrand: Brand name\\nDescription: 1 sentence description\\nDetails: 1 sentence on features"}, {"role": "user", "content": "Proline PLJW 120 Under Cabinet Range Hood - 6 Speed - 900 Max CFM Low Profile - Stainless Steel Professional Baffle Filters Dishwasher safe 3 Year Warranty Sizes include 30 36 42 and 48 inch\\n[\'Stainless Baffle Filters that are easy to remove, and the quietest 385 CFM setting in the industry. (Based on comparable size and dual local blower capacity). 900 CFM total capacity with Elegant and Efficient \\"Time Delay\\" Touch Controls. This Range Hood comes with blower and fan completely installed, and factory tested. This makes the installation one of the e

In [19]:

def make_file(start, end, filename):
    batch_file = filename
    with open(batch_file, "w", encoding="utf-8") as f:
        for i in range(start, end):
            f.write(make_jsonl(items[i]))
            f.write("\n")

In [20]:
make_file(0, 1000, "jsonl/0_1000.jsonl")

In [ ]:
import os
from groq import Groq

groq = Groq(api_key=os.environ.get("GROQ_API_KEY"))

In [ ]:

with open("jsonl/0_1000.jsonl", "rb", encoding="utf-8") as f:
    response = client.files.create(file=f, purpose="batch")
response

In [ ]:
file_id = response.id
file_id

In [ ]:
response = groq.batches.create(completion_window="24h", endpoint="/v1/chat/completions", input_file_id=file_id)
response

In [ ]:
result = groq.batches.retrieve(response.id)
result

In [ ]:
response = groq.files.content(result.output_file_id)
response.write_to_file("jsonl/batch_results.jsonl")

In [ ]:
with open("jsonl/batch_results.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        json_line = json.loads(line)
        id = int(json_line["custom_id"])
        summary = json_line["response"]["body"]["choices"][0]["message"]["content"]
        items[id].summary = summary


In [ ]:
print(items[0].full)

In [ ]:
print(items[1000].summary)

## I've put exactly this logic into a Batch class

- Divides items into groups of 1,000
- Kicks off batches for each
- Allows us to monitor and collect the results when complete

## COSTS

Using Groq, for me - this cost under $1 for the Lite dataset and under $30 for the big dataset

But you don't need to pay anything! In the next lab, you can load my pre-processed results

In [4]:
Batch.create(items, LITE_MODE)

NameError: name 'items' is not defined

In [1]:
Batch.run()

NameError: name 'Batch' is not defined

In [ ]:
Batch.fetch()

In [ ]:
for index, item in enumerate(items):
    if not item.summary:
        print(index)

In [ ]:
print(items[10234].summary)

In [ ]:
# Remove the fields that we don't need in the hub

for item in items:
    item.full = None
    item.id = None

## Push the final dataset to the hub

If lite mode, we'll only push the lite dataset

If full mode, we'll push both datasets (in case you decide to use lite later)

In [ ]:
username = "ed-donner"
full = f"{username}/items_full"
lite = f"{username}/items_lite"

if LITE_MODE:
    train = items[:20_000]
    val = items[20_000:21_000]
    test = items[21_000:]
    Item.push_to_hub(lite, train, val, test)
else:
    train = items[:800_000]
    val = items[800_000:810_000]
    test = items[810_000:]
    Item.push_to_hub(full, train, val, test)

    train_lite = train[:20_000]
    val_lite = val[:1_000]
    test_lite = test[:1_000]
    Item.push_to_hub(lite, train_lite, val_lite, test_lite)

## And here they are!

https://huggingface.co/datasets/ed-donner/items_lite

https://huggingface.co/datasets/ed-donner/items_full
